# CBTV demo: track the bundled 60-frame fixture

This notebook installs CBTV, tracks `tests/fixtures/primary60` (534 coloured particles, 60 frames, CPU only, using the
fixture's detections CSV), checks the result against the reference output, and then looks at the tracks: one
annotated frame, a few trajectories and simple statistics. The last section shows how to run it on your own video.

**Colab:** the first cell clones the repository and installs CBTV. Run locally from a checkout, the notebook uses
that checkout and installs nothing.

## 1. Install

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = os.environ.get("CBTV_REPO_URL", "https://github.com/MysteriousPenguin57/CBTV.git")

def find_repo():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "tests" / "fixtures" / "primary60").is_dir() and (d / "run_cbtv.py").exists():
            return d
    return None

ROOT = find_repo()
if ROOT is None and IN_COLAB:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "CBTV"], check=True)
    ROOT = Path("CBTV").resolve()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(ROOT), "pandas", "matplotlib"], check=True)
assert ROOT is not None, "run this notebook inside a CBTV checkout (or in Colab)"
print("CBTV checkout:", ROOT)

## 2. Track the fixture (CPU, about 1-3 minutes)

`--detections-csv` makes CBTV use the fixture's detections (the `csv` detector), so the run is fully reproducible. The default preset is `bt`.

In [ ]:
FIX = ROOT / "tests" / "fixtures" / "primary60"
OUT = Path("demo_out").resolve()
cmd = [sys.executable, str(ROOT / "run_cbtv.py"), str(FIX / "clip.mp4"),
       "--detections-csv", str(FIX / "detections.csv"), "--gpu", "0", "--output-dir", str(OUT)]
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1")
r = subprocess.run(cmd, capture_output=True, text=True, env=env)
print(r.stdout[-1500:])
assert r.returncode == 0, r.stderr[-3000:]

Check against the reference output of the `bt` preset (`golden_gpu_tracks_bt.csv`): it should be identical, byte for byte.

In [ ]:
same = (OUT / "tracks.csv").read_bytes() == (FIX / "golden_gpu_tracks_bt.csv").read_bytes()
print("identical to the reference:", same)

## 3. Look at the tracks

`tracks.csv` has one row per particle per frame: `frame, track_id, x, y, r, missed` (`missed > 0` = coasting, i.e. predicted, not detected).

In [ ]:
import pandas as pd

tr = pd.read_csv(OUT / "tracks.csv")
lengths = tr.groupby("track_id")["frame"].nunique()
stats = {
    "frames": int(tr["frame"].nunique()),
    "tracks": int(tr["track_id"].nunique()),
    "mean track length (frames)": round(float(lengths.mean()), 2),
    "tracks spanning every frame": int((lengths == tr["frame"].nunique()).sum()),
    "coasting rows (%)": round(100.0 * float((tr["missed"] > 0).mean()), 2),
}
print(stats)
tr.head()

In [ ]:
import cv2
import matplotlib
import matplotlib.pyplot as plt

FRAME = 30
cap = cv2.VideoCapture(str(FIX / "clip.mp4"))
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME)
ok, img = cap.read()
cap.release()
assert ok
rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

cmap = matplotlib.colormaps["tab20"]
at = tr[(tr["frame"] == FRAME) & (tr["missed"] == 0)]
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].imshow(rgb)
for _, p in at.iterrows():
    ax[0].add_patch(plt.Circle((p.x, p.y), p.r, fill=False, lw=0.8, color=cmap(int(p.track_id) % 20)))
ax[0].set_title(f"frame {FRAME}: {len(at)} tracked particles")

ax[1].imshow(rgb, alpha=0.35)
for tid in tr["track_id"].drop_duplicates().sample(12, random_state=0):
    t = tr[tr["track_id"] == tid].sort_values("frame")
    ax[1].plot(t.x, t.y, "-", lw=1.5, color=cmap(int(tid) % 20))
    ax[1].plot(t.x.iloc[-1], t.y.iloc[-1], "o", ms=4, color=cmap(int(tid) % 20))
ax[1].set_title("12 random trajectories (dot = last frame)")
for a in ax:
    a.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
ax = lengths.plot.hist(bins=30, figsize=(6, 3), title="track length (frames)")
ax.set_xlabel("frames"); plt.show()

## 4. Your own video

Replace the paths below and run the cell (set `RUN_OWN = True`). Without a detections CSV CBTV runs its default
detector (`ws_multi_vote`); set `--r-min/--r-max` to your particle radius range in pixels (README section 5).
Outputs: `tracks.csv`, `annotated.mp4`, `overlay.mp4`, `summary.txt` in `my_out/`.

In [ ]:
RUN_OWN = False
MY_VIDEO = "my_experiment.mp4"          # your video
MY_DETECTIONS = None                    # or "my_detections.csv" (frame,x,y,r[,score])
if RUN_OWN:
    cmd = [sys.executable, str(ROOT / "run_cbtv.py"), MY_VIDEO, "--output-dir", "my_out",
           "--r-min", "6", "--r-max", "10"]
    if MY_DETECTIONS:
        cmd += ["--detections-csv", MY_DETECTIONS]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    print(Path("my_out/summary.txt").read_text())